In [13]:
from mGST.utility_functions_comparisons import get_full_mgst_parameters_from_configuration, factorize_psd_truncated, create_4q_gst_config
from iqm.benchmarks.compressive_gst.compressive_gst import GSTConfiguration
 
from mGST.low_level_jit import cost_function_numba, cost_function_jax_mps, gradient_k_and_value_jit, gradient_all_3_and_value_jit

import jax.numpy as jnp
import jax

import numpy as np

import matplotlib.pyplot as plt

backend = "iqmfakeapollo"

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
Q4_GST = create_4q_gst_config()

K, X, E, rho, y, J, l, depth, pdim, dim_squared, n_povm, bsize, meas_samples, n, nt, rK = get_full_mgst_parameters_from_configuration(
    Q4_GST, backend, seed=42
)

2025-02-20 11:48:13,180 - iqm.benchmarks.logging_config - INFO - Now generating 2000 random GST circuits...
2025-02-20 11:48:15,292 - iqm.benchmarks.logging_config - INFO - Will transpile all 2000 circuits according to fixed physical layout
2025-02-20 11:48:15,292 - iqm.benchmarks.logging_config - INFO - Transpiling for backend IQMFakeApolloBackend with optimization level 0, sabre routing method all circuits
2025-02-20 11:48:26,350 - iqm.benchmarks.logging_config - INFO - Submitting batch with 2000 circuits corresponding to qubits [0, 1, 3, 4]
2025-02-20 11:48:26,370 - iqm.benchmarks.logging_config - INFO - Now executing the corresponding circuit batch
2025-02-20 11:48:26,393 - iqm.benchmarks.logging_config - INFO - Retrieving all counts
INFO:2025-02-20 11:49:30,891:jax._src.xla_bridge:927: Unable to initialize backend 'rocm': module 'jaxlib.xla_extension' has no attribute 'GpuAllocatorConfig'
2025-02-20 11:49:30,891 - jax._src.xla_bridge - INFO - Unable to initialize backend 'rocm': m

In [4]:
K_np = np.array(K)
E_np = np.array(E)
rho_np = np.array(rho)
y_np = np.array(y)

# creating the parameters needed for the jax version
num_povm = E.shape[0]
dim = int(jnp.sqrt(E.shape[1]))

povm_tensor = jnp.reshape(E, shape=(num_povm, dim, dim))
state = jnp.reshape(rho, shape=(dim, dim))

indices_list = [indices[indices != -1] for indices in J]

kraus_tensor = K
prob_matrix = y

state_psd = factorize_psd_truncated(psd=state, max_rank=1)
povm_psd  = factorize_psd_truncated(psd=povm_tensor, max_rank=1)

jnp.allclose(state_psd @ state_psd.T, state), jnp.allclose(povm_psd @ povm_psd.conj().transpose(0, 2, 1), povm_tensor)

(Array(True, dtype=bool), Array(True, dtype=bool))

In [10]:
%timeit cost_function_jax_mps(kraus_tensor, povm_psd, state_psd, indices_list, prob_matrix, jit=True)

Initial count: 0
at iteration 0 the new count changed to: 1
at iteration 1 the new count changed to: 2
at iteration 10 the new count changed to: 3
at iteration 91 the new count changed to: 4
at iteration 242 the new count changed to: 5
at iteration 393 the new count changed to: 6
at iteration 544 the new count changed to: 7
at iteration 696 the new count changed to: 8
at iteration 848 the new count changed to: 9
at iteration 1000 the new count changed to: 10
at iteration 1166 the new count changed to: 11
at iteration 1332 the new count changed to: 12
at iteration 1499 the new count changed to: 13
at iteration 1666 the new count changed to: 14
at iteration 1833 the new count changed to: 15
Initial count: 15
Initial count: 15
Initial count: 15
Initial count: 15
Initial count: 15
Initial count: 15
Initial count: 15
139 ms ± 3.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [11]:
%timeit gradient_k_and_value_jit(kraus_tensor, povm_psd, state_psd, indices_list, prob_matrix)

Initial count: 15
Initial count: 15
Initial count: 15
Initial count: 15
Initial count: 15
Initial count: 15
Initial count: 15
Initial count: 15
9.31 s ± 222 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [15]:
%timeit gradient_all_3_and_value_jit(kraus_tensor, povm_psd, state_psd, indices_list, prob_matrix)

Initial count: 15
Initial count: 15
Initial count: 15
Initial count: 15
Initial count: 15
Initial count: 15
Initial count: 15
Initial count: 15
10.2 s ± 248 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [18]:
10.2 * 100 /60

16.999999999999996